## User–Item Bias Recommender (Baseline C)

This notebook implements the **user–item bias** baseline model:

\[
\hat r_{ui} = \mu + b_u + b_i
\]

- Trains **only** on `train_reviews_santa_barbara.csv`
- Evaluates **only** on `test_reviews_santa_barbara.csv`
- For each user, recommendations are drawn from **restaurants not seen in train**
- Final cell reports required metrics: **Hit@10/20/30**, **NDCG@10/20/30**, **RMSE/MAE/R2**


In [1]:
from __future__ import annotations

import math
from dataclasses import dataclass
from typing import Dict, Iterable, List, Sequence, Tuple

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


### Load data

Expected files in the project folder:
- `restaurants_santa_barbara.csv`
- `train_reviews_santa_barbara.csv`
- `test_reviews_santa_barbara.csv`


In [2]:
TRAIN_PATH = "train_reviews_santa_barbara.csv"
TEST_PATH = "test_reviews_santa_barbara.csv"
ITEMS_PATH = "restaurants_santa_barbara.csv"

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
items = pd.read_csv(ITEMS_PATH)

required_review_cols = {"user_id", "business_id", "stars"}
assert required_review_cols.issubset(train.columns), f"train missing cols: {required_review_cols - set(train.columns)}"
assert required_review_cols.issubset(test.columns), f"test missing cols: {required_review_cols - set(test.columns)}"
assert "business_id" in items.columns, "restaurants CSV must contain business_id"

train = train[["user_id", "business_id", "stars"]].copy()
test = test[["user_id", "business_id", "stars"]].copy()

train["user_id"] = train["user_id"].astype(str)
train["business_id"] = train["business_id"].astype(str)
test["user_id"] = test["user_id"].astype(str)
test["business_id"] = test["business_id"].astype(str)
items["business_id"] = items["business_id"].astype(str)

train["stars"] = train["stars"].astype(float)
test["stars"] = test["stars"].astype(float)

train.head()

,user_id,business_id,stars
0,-0EcgtUXe1rzrkmdih_tYg,VgAKmXE8B7J0I_O_R13UKQ,4.0
1,-0EcgtUXe1rzrkmdih_tYg,YrNtBUOUOYwmRZ_UVwH8iQ,4.0
2,-0EcgtUXe1rzrkmdih_tYg,Aes-0Q_guDeYewMapFs_vg,4.0
3,-0EcgtUXe1rzrkmdih_tYg,9ugpNKKhnYRa51qXoxUw_A,3.0
4,-0EcgtUXe1rzrkmdih_tYg,DOfiulOub9hVPBCtiDl9Fw,3.0


### Bias model implementation

We fit user and item biases by **regularized coordinate descent**:
- Initialize \(b_u = 0, b_i = 0\)
- Alternating updates:
  - \(b_u \leftarrow \frac{\sum_{i\in I(u)} (r_{ui} - \mu - b_i)}{\lambda + |I(u)|}\)
  - \(b_i \leftarrow \frac{\sum_{u\in U(i)} (r_{ui} - \mu - b_u)}{\lambda + |U(i)|}\)


In [3]:
@dataclass
class BiasModel:
    """User–Item bias model: r_hat(u,i) = mu + b_u + b_i.

    Attributes
    ----------
    mu:
        Global mean rating from the training set.
    b_u:
        Mapping user_id -> user bias.
    b_i:
        Mapping business_id -> item bias.
    """

    mu: float
    b_u: Dict[str, float]
    b_i: Dict[str, float]

    def predict_rating(self, user_id: str, business_id: str) -> float:
        """Predict a rating for a (user, item) pair.

        Parameters
        ----------
        user_id:
            Yelp user id.
        business_id:
            Yelp business id.

        Returns
        -------
        float
            Predicted rating.
        """
        pred = self.mu + self.b_u.get(user_id, 0.0) + self.b_i.get(business_id, 0.0)
        return float(np.clip(pred, 1.0, 5.0))


def fit_bias_model(
    train_df: pd.DataFrame,
    reg: float = 10.0,
    n_iters: int = 15,
    verbose: bool = True,
) -> BiasModel:
    """Fit a user–item bias model using regularized coordinate descent.

    Parameters
    ----------
    train_df:
        DataFrame with columns [user_id, business_id, stars].
    reg:
        L2-style regularization strength (higher = stronger shrinkage).
    n_iters:
        Number of alternating update iterations.
    verbose:
        If True, prints training RMSE each iteration.

    Returns
    -------
    BiasModel
        Trained bias model.
    """
    df = train_df[["user_id", "business_id", "stars"]].copy()
    mu = float(df["stars"].mean())

    # Group indices for fast residual aggregation
    by_user = df.groupby("user_id")
    by_item = df.groupby("business_id")

    users = df["user_id"].unique().tolist()
    items_ = df["business_id"].unique().tolist()

    b_u: Dict[str, float] = {u: 0.0 for u in users}
    b_i: Dict[str, float] = {i: 0.0 for i in items_}

    for it in range(n_iters):
        # Update user biases
        for u, g in by_user:
            # g has columns user_id, business_id, stars
            r = g["stars"].to_numpy(dtype=float)
            i_ids = g["business_id"].tolist()
            resid = r - mu - np.array([b_i.get(i, 0.0) for i in i_ids], dtype=float)
            b_u[u] = float(resid.sum() / (reg + len(g)))

        # Update item biases
        for i, g in by_item:
            r = g["stars"].to_numpy(dtype=float)
            u_ids = g["user_id"].tolist()
            resid = r - mu - np.array([b_u.get(u, 0.0) for u in u_ids], dtype=float)
            b_i[i] = float(resid.sum() / (reg + len(g)))

        if verbose:
            preds = df.apply(lambda row: mu + b_u[row["user_id"]] + b_i[row["business_id"]], axis=1)
            mse = mean_squared_error(df["stars"].to_numpy(dtype=float), preds.to_numpy(dtype=float))
            rmse = float(np.sqrt(mse))
            print(f"iter={it+1:02d}/{n_iters} train_RMSE={rmse:.4f}")

    return BiasModel(mu=mu, b_u=b_u, b_i=b_i)


### Ranking metrics helpers (Hit@K / NDCG@K)

The test set has **one held-out interaction per user**, so:
- **Hit@K** is 1 if the held-out item is in the top-K list, else 0
- **NDCG@K** is \(1/\log_2(rank+1)\) if present, else 0


In [4]:
def hit_at_k(recommended: Sequence[str], target_item: str, k: int) -> float:
    """Compute Hit@K for one user.

    Parameters
    ----------
    recommended:
        Ranked list of recommended business_ids (highest score first).
    target_item:
        The held-out test business_id.
    k:
        Cutoff.

    Returns
    -------
    float
        1.0 if target appears in top-K, else 0.0.
    """
    return 1.0 if target_item in recommended[:k] else 0.0


def ndcg_at_k(recommended: Sequence[str], target_item: str, k: int) -> float:
    """Compute NDCG@K for one user with a single relevant item.

    Parameters
    ----------
    recommended:
        Ranked list of recommended business_ids (highest score first).
    target_item:
        The held-out test business_id.
    k:
        Cutoff.

    Returns
    -------
    float
        NDCG@K value (0.0 if not present).
    """
    topk = recommended[:k]
    if target_item not in topk:
        return 0.0
    rank = topk.index(target_item) + 1  # 1-based
    return float(1.0 / math.log2(rank + 1))


### Recommendation generation

For each test user, we rank **all candidate restaurants** (all restaurants in the metadata file) **excluding** anything the user interacted with in the training set.


In [5]:
def build_train_history(train_df: pd.DataFrame) -> Dict[str, set]:
    """Build a user -> set(items) interaction history from training data.

    Parameters
    ----------
    train_df:
        DataFrame with columns [user_id, business_id].

    Returns
    -------
    dict
        Mapping user_id -> set of business_id seen in train.
    """
    hist: Dict[str, set] = {}
    for u, i in zip(train_df["user_id"].tolist(), train_df["business_id"].tolist()):
        hist.setdefault(u, set()).add(i)
    return hist


def recommend_top_k(
    model: BiasModel,
    user_id: str,
    candidate_items: Sequence[str],
    seen_in_train: set,
    k: int,
) -> List[str]:
    """Generate top-K recommendations for a user.

    Parameters
    ----------
    model:
        Trained bias model.
    user_id:
        Yelp user id.
    candidate_items:
        All candidate business_ids that can be recommended.
    seen_in_train:
        Items the user interacted with in training and must be excluded.
    k:
        Number of recommendations.

    Returns
    -------
    list
        Top-K business_ids ranked by predicted score.
    """
    # Score all unseen candidates
    scores: List[Tuple[str, float]] = []
    u_bias = model.b_u.get(user_id, 0.0)
    for i in candidate_items:
        if i in seen_in_train:
            continue
        score = model.mu + u_bias + model.b_i.get(i, 0.0)
        scores.append((i, float(score)))

    # Sort by score descending, tie-break by item id for determinism
    scores.sort(key=lambda x: (-x[1], x[0]))
    return [i for i, _ in scores[:k]]


### Train the model


In [ ]:
model = fit_bias_model(train, reg=10.0, n_iters=15, verbose=True)
model

iter=01/15 train_RMSE=1.0002
iter=02/15 train_RMSE=0.9990
iter=03/15 train_RMSE=0.9991
iter=04/15 train_RMSE=0.9991


## Final evaluation (required deliverable)

**This must be the final cell of the notebook** per project instructions.

- Ranking eval: For each user in the test set, rank all candidate restaurants excluding train history.
- Rating eval: Predict the held-out test star rating.


In [ ]:
train_hist = build_train_history(train)
all_business_ids = items["business_id"].astype(str).tolist()

Ks = [10, 20, 30]
hit_sums = {k: 0.0 for k in Ks}
ndcg_sums = {k: 0.0 for k in Ks}

y_true: List[float] = []
y_pred: List[float] = []

for row in test.itertuples(index=False):
    u = str(row.user_id)
    target_i = str(row.business_id)
    true_r = float(row.stars)

    seen = train_hist.get(u, set())
    recs = recommend_top_k(model, user_id=u, candidate_items=all_business_ids, seen_in_train=seen, k=max(Ks))

    for k in Ks:
        hit_sums[k] += hit_at_k(recs, target_i, k)
        ndcg_sums[k] += ndcg_at_k(recs, target_i, k)

    y_true.append(true_r)
    y_pred.append(model.predict_rating(u, target_i))

n_users = len(test)
hit = {k: hit_sums[k] / n_users for k in Ks}
ndcg = {k: ndcg_sums[k] / n_users for k in Ks}

mse = mean_squared_error(y_true, y_pred)
rmse = float(np.sqrt(mse))
mae = mean_absolute_error(y_true, y_pred)
r2 = r2_score(y_true, y_pred)

print("User–Item Bias Recommender — Test Metrics")
print("Ranking metrics:")
for k in Ks:
    print(f"  Hit@{k}:  {hit[k]:.4f}    NDCG@{k}: {ndcg[k]:.4f}")

print("\nRating prediction metrics:")
print(f"  RMSE: {rmse:.4f}")
print(f"  MAE:  {mae:.4f}")
print(f"  R2:   {r2:.4f}")


User–Item Bias Recommender — Test Metrics
Ranking metrics:
  Hit@10:  0.0237    NDCG@10: 0.0108
  Hit@20:  0.0412    NDCG@20: 0.0151
  Hit@30:  0.0558    NDCG@30: 0.0181

Rating prediction metrics:
  RMSE: 1.1451
  MAE:  0.9053
  R2:   0.1578
